# 06 · Per-subject confidence intervals and leave-subjects-out (session 4)
**Part A** evaluates the session-2 checkpoint (`patch_both_prev0_K0_qat5_v2.pt`) per subject, with 95 % cluster-bootstrap
confidence intervals (whole test runs are resampled, because windows inside a run are correlated). It answers:
does the delta head's small gain (`delta_gain = RMSE(abs_smoothed) − RMSE(kalman)`) hold for most subjects, or is it one subject?

**Part B** trains 5 folds, each on 8 subjects, and tests on the held-out 2 subjects' test repetitions (2 and 5).
Held-out subjects are normalised with constants from their own training repetitions: a short calibration
recording, but no retraining. Same architecture as session 2, float training (no QAT) to save time.
Folds resume: if the session dies, attach this notebook's output as a dataset and re-run; finished folds are reused.

**Before running:** re-upload `emg_common.py` (adds `per_subject_eval`, `run_loso`). Attach the session-1 dataset
(`db2_processed/`) and the session-2 output (the checkpoint). GPU on. Expected time: about 3 to 3.5 hours.
Click **Save Version** at the end.

In [ ]:
import os, sys, glob, json
for p in ['/kaggle/usr/lib/notebooks/quddusikashaf/emg_common', '/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '.', '..', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
assert hasattr(ec, 'per_subject_eval') and hasattr(ec, 'run_loso'), 're-upload the updated emg_common.py utility script'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')
N_BOOT = int(os.environ.get('EMG_NBOOT', 2000)); N_FOLDS = int(os.environ.get('EMG_FOLDS', 5))
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
OUT = f'{WORK}/subject_eval'; os.makedirs(OUT, exist_ok=True)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| emg_common', ec.__file__)

In [ ]:
candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/**/db2_processed', recursive=True))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'attach the session-1 dataset that contains db2_processed/'
print(len(npz_files), 'subjects:', [ec.subject_name(f) for f in npz_files])

## Part A · per-subject results for the session-2 model, with 95 % bootstrap intervals

In [ ]:
ckpts = sorted(glob.glob('/kaggle/input/**/patch_both_prev0_K0_qat5_v2.pt', recursive=True))
CKPT = os.environ.get('EMG_CKPT') or (ckpts[0] if ckpts else None)
assert CKPT, 'attach the session-2 output (outputs/patch_both_prev0_K0_qat5_v2.pt)'
device = ec.get_device()
model, cfg, extra = ec.load_checkpoint(CKPT, device); model.eval()
print('checkpoint', CKPT, '| phase', extra.get('phase'))
# filter settings tuned in session 3: fusion (q, r) = (1e-5, 3e-3); smoothing only (1e-5, 1e-3); seeded from the model
within = ec.per_subject_eval(model, npz_files, cfg, device, init='abs', kalman_qr=(1e-5, 3e-3), smooth_qr=(1e-5, 1e-3),
                             n_boot=N_BOOT, tag='within_subject')
within.to_csv(f'{OUT}/within_subject.csv', index=False)
cols = ['subject', 'runs', 'windows', 'abs_only_r2', 'abs_smoothed_r2', 'kalman_r2', 'kalman_r2_lo', 'kalman_r2_hi',
        'kalman_rmse', 'delta_gain', 'delta_gain_lo', 'delta_gain_hi', 'delta_gain_p_le0']
display(within[cols].round(4))
subj = within[within['subject'] != 'ALL']
print(f"delta head helps in {(subj['delta_gain'] > 0).sum()} of {len(subj)} subjects; "
      f"CI excludes zero (gain > 0) in {(subj['delta_gain_lo'] > 0).sum()}; pooled gain "
      f"{within.iloc[-1]['delta_gain']:.4f} [{within.iloc[-1]['delta_gain_lo']:.4f}, {within.iloc[-1]['delta_gain_hi']:.4f}]")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
y = np.arange(len(within))
ax[0].errorbar(within['kalman_r2'], y, xerr=[within['kalman_r2'] - within['kalman_r2_lo'], within['kalman_r2_hi'] - within['kalman_r2']], fmt='o', capsize=3)
ax[0].scatter(within['abs_only_r2'], y, marker='x', color='gray', label='abs head only')
ax[0].set_yticks(y); ax[0].set_yticklabels(within['subject']); ax[0].set_xlabel('R² (fused), 95 % CI'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].errorbar(within['delta_gain'], y, xerr=[within['delta_gain'] - within['delta_gain_lo'], within['delta_gain_hi'] - within['delta_gain']], fmt='o', capsize=3, color='C2')
ax[1].axvline(0, color='k', lw=1); ax[1].set_yticks(y); ax[1].set_yticklabels(within['subject'])
ax[1].set_xlabel('delta-head gain: RMSE(smoothed) − RMSE(fused), 95 % CI'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/within_subject_ci.png', dpi=120); plt.show()

## Part B · leave-subjects-out (5 folds × 2 held-out subjects)

In [ ]:
loso_cfg = ec.Config.from_dict({**cfg.to_dict(), 'save_dir': f'{WORK}/outputs', 'qat_epochs': 0, 'epochs': 40, 'patience': 8,
                                'use_wandb': False})
if EPOCHS_OVERRIDE is not None: loso_cfg.epochs = EPOCHS_OVERRIDE
loso_cfg.run_name = 'loso_patch_both'
reuse = sorted({os.path.dirname(p) for p in glob.glob('/kaggle/input/**/loso_patch_both_fold*.pt', recursive=True)})
print('fold checkpoints available for reuse:', reuse)
loso = ec.run_loso(loso_cfg, npz_files, n_folds=N_FOLDS, out_dir=f'{WORK}/outputs/loso', reuse_dirs=reuse,
                   init='abs', kalman_qr=(1e-5, 3e-3), smooth_qr=(1e-5, 1e-3), n_boot=N_BOOT)
loso.to_csv(f'{OUT}/loso.csv', index=False)

In [ ]:
# ---- within-subject vs leave-subjects-out on the SAME test windows ----
w = within[within['subject'] != 'ALL'].set_index('subject'); l = loso.set_index('subject')
both = w[['kalman_r2', 'kalman_rmse', 'abs_only_r2']].join(l[['setting', 'kalman_r2', 'kalman_rmse', 'abs_only_r2', 'delta_gain']],
                                                          lsuffix='_within', rsuffix='_loso', how='inner')
both['r2_drop'] = both['kalman_r2_within'] - both['kalman_r2_loso']
display(both.round(4))
summary = pd.DataFrame({
    'within_subject': [w['kalman_r2'].mean(), w['kalman_r2'].std(), w['kalman_rmse'].mean(), w['abs_only_r2'].mean()],
    'leave_subjects_out': [l['kalman_r2'].mean(), l['kalman_r2'].std(), l['kalman_rmse'].mean(), l['abs_only_r2'].mean()]},
    index=['fused R² (mean over subjects)', 'fused R² (std over subjects)', 'fused RMSE (mean)', 'abs-head R² (mean)'])
display(summary.round(4)); summary.to_csv(f'{OUT}/summary.csv')
fig, ax = plt.subplots(figsize=(10, 5)); x = np.arange(len(both))
ax.bar(x - 0.2, both['kalman_r2_within'], 0.4, label='within-subject (session-2 model)')
ax.bar(x + 0.2, both['kalman_r2_loso'], 0.4, label='leave-subjects-out')
ax.set_xticks(x); ax.set_xticklabels(both.index, rotation=30); ax.set_ylabel('fused R²'); ax.legend(); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'{OUT}/within_vs_loso.png', dpi=120); plt.show()
json.dump({'checkpoint': CKPT, 'n_boot': N_BOOT, 'n_folds': N_FOLDS, 'summary': summary.to_dict()}, open(f'{OUT}/session4_summary.json', 'w'), indent=2)

### Reading the results
* **Part A:** if `delta_gain_lo > 0` for most subjects, the delta head's contribution is robust; if the pooled interval spans zero, report smoothing only.
* **Part B:** the drop from within-subject to leave-subjects-out R² is the cost of a new user with calibration but no retraining.
  It is the number to quote for "generalises to new users".